<a href="https://colab.research.google.com/github/2303A52151/reinforcement_learning_2151/blob/main/lab_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Part 1: Greedy Algorithm using Arithmetic Mean Method

In [1]:
# Define the machines and their observed rewards
rewards_dict = {
    'A': [3, 2, 4],
    'B': [6, 7],
    'C': [5, 4, 6]
}

q_values = {}

# Task 1: Calculate the initial estimated value Q(a) of each machine using the arithmetic mean.
for machine, rewards in rewards_dict.items():
    if len(rewards) > 0:
        q_values[machine] = sum(rewards) / len(rewards)
    else:
        q_values[machine] = 0.0

# Display the results according to the Expected Output Format
for machine, rewards in rewards_dict.items():
    print(f"Machine {machine}")
    print(f"Rewards: {rewards}")
    print(f"Q({machine}) = {q_values[machine]:.2f}\n")

# Task 2: Identify which machine the Greedy algorithm will select.
selected_machine = max(q_values, key=q_values.get)
print(f"Greedy Selected Machine: {selected_machine}")

Machine A
Rewards: [3, 2, 4]
Q(A) = 3.00

Machine B
Rewards: [6, 7]
Q(B) = 6.50

Machine C
Rewards: [5, 4, 6]
Q(C) = 5.00

Greedy Selected Machine: B


### Part 2: Greedy Algorithm using Incremental Update Method

In [ ]:
# Task 1: Initialize Q(a) = 0 and N(a) = 0 for all machines
machines = ['A', 'B', 'C']
Q = {m: 0.0 for m in machines}
N = {m: 0 for m in machines}

# Observed sequences of plays/rewards to simulate sequentially
# Format: (machine, reward)
plays = [
    ('A', 3), ('A', 2), ('A', 4),
    ('B', 6), ('B', 7),
    ('C', 5), ('C', 4), ('C', 6)
]

# Task 2 & 3: Update After Every Play
print("Incremental Update Simulation:\n" + "="*40)
for step, (machine, reward) in enumerate(plays, 1):
    # 3. Increment N(a)
    N[machine] += 1

    # 4. Update Q(a) using the incremental formula
    Q_old = Q[machine]
    Q[machine] = Q_old + (reward - Q_old) / N[machine]

    # 6. Identify the machine with the highest current Q(a)
    best_machine = max(Q, key=Q.get)

    print(f"Play {step}: Played Machine {machine}, Observed Reward = {reward}")
    print(f"Updated counts N: {N}")
    print(f"Updated estimates Q: {{k: round(v, 2) for k, v in Q.items()}}")
    print(f"Current Greedy Choice: Machine {best_machine}\n" + "-"*40)

Incremental Update Simulation:
Play 1: Played Machine A, Observed Reward = 3
Updated counts N: {'A': 1, 'B': 0, 'C': 0}
Updated estimates Q: {k: round(v, 2) for k, v in Q.items()}
Current Greedy Choice: Machine A
----------------------------------------
Play 2: Played Machine A, Observed Reward = 2
Updated counts N: {'A': 2, 'B': 0, 'C': 0}
Updated estimates Q: {k: round(v, 2) for k, v in Q.items()}
Current Greedy Choice: Machine A
----------------------------------------
Play 3: Played Machine A, Observed Reward = 4
Updated counts N: {'A': 3, 'B': 0, 'C': 0}
Updated estimates Q: {k: round(v, 2) for k, v in Q.items()}
Current Greedy Choice: Machine A
----------------------------------------
Play 4: Played Machine B, Observed Reward = 6
Updated counts N: {'A': 3, 'B': 1, 'C': 0}
Updated estimates Q: {k: round(v, 2) for k, v in Q.items()}
Current Greedy Choice: Machine B
----------------------------------------
Play 5: Played Machine B, Observed Reward = 7
Updated counts N: {'A': 3, 'B':

### Part 3: Modified $\epsilon$-Greedy Strategy for Multi-Armed Bandit

In [2]:
import random

# Part A – Machine Initialization
machines = ['A', 'B', 'C']
epsilon = 0.2
N_plays = 10

# Part B – Initialize tables
N = {m: 0 for m in machines}
R = {m: [] for m in machines}  # Collected rewards list
Q = {m: 0.0 for m in machines} # Estimated Q-values

# Simulated Reward Sequences from the suggested experiment pattern
reward_patterns = {
    'A': [3, 4, 2, 5],
    'B': [6, 7, 5, 8],
    'C': [4, 3, 5, 4]
}
# Track the current index of the simulated rewards for each machine
reward_indices = {m: 0 for m in machines}

def get_simulated_reward(machine):
    """Gets the next reward from the predefined pattern for the chosen machine."""
    idx = reward_indices[machine]
    if idx < len(reward_patterns[machine]):
        reward = reward_patterns[machine][idx]
        reward_indices[machine] += 1
    else:
        # Fallback if a machine is played more than 4 times
        reward = reward_patterns[machine][-1]
    return reward

# Simulation loop
print(f"Simulation starting with epsilon = {epsilon}\n" + "="*50)
total_reward_received = 0

# Seed for reproducibility of random decisions (r < epsilon)
random.seed(42)

for play in range(1, N_plays + 1):
    # 1. Generate a random number r between 0 and 1
    r = random.random()

    # Identify current best machine (for exploration exclusion)
    best_machine = max(Q, key=Q.get)

    # 2 & 3. Decision making
    if r < epsilon:
        # Exploration: Select randomly except the current highest Q machine
        explorable_machines = [m for m in machines if m != best_machine]
        # Fallback to all if somehow none are left (should not happen with 3 machines)
        selected_machine = random.choice(explorable_machines if explorable_machines else machines)
        decision_type = "Exploration"
    else:
        # Exploitation: Select machine with highest Q-value
        selected_machine = best_machine
        decision_type = "Exploitation"

    # Part D – Reward Collection
    reward = get_simulated_reward(selected_machine)
    total_reward_received += reward

    # Part E – Update tables using incremental sample-average
    N[selected_machine] += 1
    R[selected_machine].append(reward)

    Q_old = Q[selected_machine]
    Q[selected_machine] = Q_old + (reward - Q_old) / N[selected_machine]

    print(f"Play {play} | Decision: {decision_type} (r={r:.3f})")
    print(f"Selected Machine: {selected_machine} -> Reward obtained: {reward}")
    print(f"Updated N: {N}")
    print(f"Updated Q: {{k: round(v, 3) for k, v in Q.items()}}")
    print("-"*50)

# Part F – Display Final Results
print("\n" + "="*20 + " FINAL TABLES WITH ANALYSIS " + "="*20)
print(f"Participation Count (N): {N}")
print(f"Collected Rewards   (R): {R}")
print(f"Final Q-Estimates   (Q): {{k: round(v, 3) for k, v in Q.items()}}")
print(f"Total Reward Player Received: {total_reward_received}")

Simulation starting with epsilon = 0.2
Play 1 | Decision: Exploitation (r=0.639)
Selected Machine: A -> Reward obtained: 3
Updated N: {'A': 1, 'B': 0, 'C': 0}
Updated Q: {k: round(v, 3) for k, v in Q.items()}
--------------------------------------------------
Play 2 | Decision: Exploration (r=0.025)
Selected Machine: C -> Reward obtained: 4
Updated N: {'A': 1, 'B': 0, 'C': 1}
Updated Q: {k: round(v, 3) for k, v in Q.items()}
--------------------------------------------------
Play 3 | Decision: Exploitation (r=0.245)
Selected Machine: C -> Reward obtained: 3
Updated N: {'A': 1, 'B': 0, 'C': 2}
Updated Q: {k: round(v, 3) for k, v in Q.items()}
--------------------------------------------------
Play 4 | Decision: Exploration (r=0.140)
Selected Machine: A -> Reward obtained: 4
Updated N: {'A': 2, 'B': 0, 'C': 2}
Updated Q: {k: round(v, 3) for k, v in Q.items()}
--------------------------------------------------
Play 5 | Decision: Exploitation (r=0.677)
Selected Machine: A -> Reward obtaine

### Part 4: Theoretical Calculations and Analysis

#### 1. Calculations for the first 2 plays of the simulation:
* **Play 1**:
  * Initial state: $Q(A)=0, Q(B)=0, Q(C)=0$. All are tied, so maximum is 'A'.
  * $r = 0.639 \ge \epsilon$ ($0.2$): Exploitation chosen. Selected Machine: **A**.
  * Reward pattern for A: First element is **3**.
  * Update:
    * $N(A) = 1$
    * $Q(A) = 0 + \frac{3 - 0}{1} = 3.0$

* **Play 2**:
  * Current state: $Q(A)=3.0, Q(B)=0, Q(C)=0$. Max is 'A'.
  * $r = 0.025 < \epsilon$ ($0.2$): Exploration chosen. Exclude 'A' (the max Q). Explorable: ['B', 'C']. Randomly selected: **B**.
  * Reward pattern for B: First element is **6**.
  * Update:
    * $N(B) = 1$
    * $Q(B) = 0 + \frac{6 - 0}{1} = 6.0$

#### 2. Total Reward Summary:
* The sum of all rewards collected across the 10 plays is dynamically tracked by `total_reward_received` in the Python simulation.